importing all libraries

In [0]:
from pyspark.sql.functions import *


loading the source file

In [0]:
file_path="/Volumes/grocery_delivery_project/source/initial_files/grocery_delivery_initial.csv"

reading the source file and loading it in a initial data frame called grocery_df

In [0]:
grocery_df=spark.read\
    .format("csv")\
        .option("header",True)\
            .option("inferSchema",True)\
                 .csv(file_path)

In [0]:
display(grocery_df)

In [0]:
grocery_df.count()

In [0]:
grocery_df.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("grocery_delivery_project.bronze.grocery_source")

In [0]:
grocery_df.printSchema()

In [0]:
grocery_df.select([
     sum(col(c).isNull().cast("int")).alias(c)
                   for c in grocery_df.columns
                   
]).show()


In [0]:
grocery_df.filter(
    col("order_id").isNull()
).show()

grocery_df.filter(
    col("customer_name").isNull()
).show()

grocery_df.filter(
    col("category").isNull()
).show()

In [0]:
grocery_df.filter(
    (col("quantity")<=0)
).show()

grocery_df.filter(
    (col("unit_price")<=0)
).show()

grocery_df.filter(
    (col("discount")<0)
).show()

In [0]:
grocery_df.filter(
    (col("delivery_status")=="INVALID_STATUS")
).show()

In [0]:
invalid_records=(
        (col("order_id").isNull())|
        col("customer_name").isNull()|
         col("category").isNull()|
         (col("quantity")<=0)|
         (col("unit_price")<=0)|
          (col("discount")<0)|
          (col("delivery_status")=="INVALID_STATUS")

)

In [0]:
bad_records=grocery_df.filter(
    invalid_records
)

In [0]:
bad_records.display()

In [0]:
bad_records.count()

In [0]:
good_records=grocery_df.filter(
    ~invalid_records
)
good_records.count()
good_records.display()


In [0]:
good_records.write\
    .mode("overwrite")\
    .format("delta")\
.saveAsTable("grocery_delivery_project.bronze.grocery_with_duplicate")

In [0]:
good_records.count()

In [0]:
if(bad_records.count()+good_records.count()==grocery_df.count()):
  print("the count is correct")
else:
    print("the count is not correct")

In [0]:
bad_records.write\
    .format("delta")\
    .mode("overwrite")\
        .saveAsTable("grocery_delivery_project.bronze.bad_records")

In [0]:
display(
    spark.read.table("grocery_delivery_project.bronze.bad_records")
)

In [0]:
cleaned_df=good_records.dropDuplicates(["order_id"])

display(cleaned_df)

cleaned_df.count()

In [0]:
print("dupliacate count:",good_records.count()-cleaned_df.count())

In [0]:
print("final clean record count:",cleaned_df.count())

In [0]:
cleaned_df.write\
    .mode("overwrite")\
    .format("delta")\
        .saveAsTable("grocery_delivery_project.bronze.bronze_table")

In [0]:
display(
    spark.read.table("grocery_delivery_project.bronze.bronze_table")
)